# Combined Debiasing: Data Scrubbing + GroupDRO

In [5]:
%pip install torch transformers datasets tqdm joblib pandas numpy scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import json
import re
import numpy as np
import pandas as pd
import torch
import joblib
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import DataLoader
from tqdm import tqdm
from dataclasses import dataclass
from typing import Any, Dict, List
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Проверка железа
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    print('Apple Silicon MPS')
else:
    device = torch.device('cpu')
    print('CPU mode')

print(f'Device: {device}')

Apple Silicon MPS
Device: mps


In [ ]:
# === CONFIG ===
GROUPDRO_ETA = 0.05
USE_SQRT_REWEIGHT = True
NUM_EPOCHS = 10
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
MAX_LENGTH = 128

# === PATHS ===
BASE_DIR = Path('.')
possible_data_dirs = [
    BASE_DIR / 'data' / 'processed',
    BASE_DIR / 'data',
    BASE_DIR,
]

PROCESSED_DIR = None
for d in possible_data_dirs:
    if (d / 'train.csv').exists():
        PROCESSED_DIR = d
        break

if PROCESSED_DIR is None:
    raise FileNotFoundError('train.csv not found!')

print(f'Data: {PROCESSED_DIR}')

OUTPUT_DIR = BASE_DIR / 'models' / 'combined_scrubbing_groupdro'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output: {OUTPUT_DIR}')

Data: ../data/processed
Output: ../output/combined_scrubbing_groupdro


In [8]:
# === LOAD DATA ===
df_train = pd.read_csv(PROCESSED_DIR / 'train.csv')
df_val = pd.read_csv(PROCESSED_DIR / 'val.csv')
df_test = pd.read_csv(PROCESSED_DIR / 'test.csv')

mapping_file = PROCESSED_DIR / 'label_to_supercategory_v1.csv'
if mapping_file.exists():
    mapping = pd.read_csv(mapping_file)
    label_to_supercat = dict(zip(mapping['label'], mapping['supercategory']))
    for df in [df_train, df_val, df_test]:
        df['supercategory'] = df['label'].map(label_to_supercat)
    target_col = 'supercategory'
else:
    target_col = 'label'

le = LabelEncoder()
df_train['y'] = le.fit_transform(df_train[target_col])
df_val['y'] = le.transform(df_val[target_col])
df_test['y'] = le.transform(df_test[target_col])

num_labels = len(le.classes_)
print(f'Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}, Labels: {num_labels}')

Train: 16530, Val: 5510, Test: 5510, Labels: 9


In [9]:
# === DATA SCRUBBING ===
SENSITIVE_WORDS = [
    'москва', 'московская', 'мск', 'санкт-петербург', 'петербург', 'спб', 'питер',
    'новосибирск', 'екатеринбург', 'казань', 'нижний новгород', 'челябинск', 'самара',
    'омск', 'ростов-на-дону', 'уфа', 'красноярск', 'воронеж', 'пермь', 'волгоград',
    'краснодар', 'саратов', 'тюмень', 'тольятти', 'ижевск', 'барнаул', 'ульяновск',
    'иркутск', 'хабаровск', 'ярославль', 'владивосток', 'махачкала', 'томск',
    'область', 'край', 'республика', 'регион',
    'пенсионер', 'пенсионерка', 'пенсия', 'студент', 'студентка',
    'выпускник', 'выпускница', 'молодой', 'молодая', 'junior', 'senior',
]

ALL_SENSITIVE = set(w.lower() for w in SENSITIVE_WORDS)

def scrub_text(text, sensitive_words, mask_token='[MASK]'):
    if pd.isna(text):
        return ''
    result = str(text)
    for word in sorted(sensitive_words, key=len, reverse=True):
        pattern = re.compile(re.escape(word), re.IGNORECASE)
        result = pattern.sub(mask_token, result)
    return result

print('Scrubbing...')
text_col = 'resume_text' if 'resume_text' in df_train.columns else df_train.columns[0]
for df in [df_train, df_val, df_test]:
    df['text_scrubbed'] = df[text_col].apply(lambda x: scrub_text(x, ALL_SENSITIVE))
print('Done!')

Scrubbing...
Done!


In [10]:
# === GROUPDRO SETUP ===
group_col = 'city_group' if 'city_group' in df_train.columns else None

if group_col:
    city_to_id = {c: i for i, c in enumerate(sorted(df_train[group_col].unique()))}
    num_groups = len(city_to_id)
    df_train['group_id'] = df_train[group_col].map(city_to_id).astype(int)
    df_val['group_id'] = df_val[group_col].map(city_to_id).fillna(0).astype(int)
    df_test['group_id'] = df_test[group_col].map(city_to_id).fillna(0).astype(int)
    
    if USE_SQRT_REWEIGHT:
        city_counts = df_train[group_col].value_counts()
        raw_w = 1.0 / np.sqrt(city_counts)
        city_weight_map = (raw_w / raw_w.mean()).to_dict()
        df_train['sample_weight'] = df_train[group_col].map(city_weight_map).astype(float)
    else:
        df_train['sample_weight'] = 1.0
else:
    num_groups = 1
    for df in [df_train, df_val, df_test]:
        df['group_id'] = 0
    df_train['sample_weight'] = 1.0

print(f'Groups: {num_groups}')

Groups: 41


In [11]:
# === TOKENIZATION ===
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

print('Tokenizing...')

def prepare_data(df, include_weights=False):
    records = []
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        enc = tokenizer(
            row['text_scrubbed'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt'
        )
        record = {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': row['y'],
            'group_id': row['group_id'],
        }
        if include_weights:
            record['sample_weight'] = row['sample_weight']
        records.append(record)
    return records

train_data = prepare_data(df_train, include_weights=True)
val_data = prepare_data(df_val)
test_data = prepare_data(df_test)

print(f'Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}')

Tokenizing...


100%|██████████| 5510/5510 [00:30<00:00, 182.34it/s]

Train: 16530, Val: 5510, Test: 5510


In [12]:
# === DATASET & COLLATOR ===
class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

@dataclass
class GroupDROCollator:
    def __call__(self, features):
        batch = {
            'input_ids': torch.stack([f['input_ids'] for f in features]),
            'attention_mask': torch.stack([f['attention_mask'] for f in features]),
            'labels': torch.tensor([f['labels'] for f in features], dtype=torch.long),
            'group_id': torch.tensor([f['group_id'] for f in features], dtype=torch.long),
        }
        if 'sample_weight' in features[0]:
            batch['sample_weight'] = torch.tensor([f['sample_weight'] for f in features], dtype=torch.float)
        return batch

train_dataset = SimpleDataset(train_data)
val_dataset = SimpleDataset(val_data)
test_dataset = SimpleDataset(test_data)
print('Datasets ready')

Datasets ready


In [13]:
# === GROUPDRO TRAINER ===
class GroupDROTrainer(Trainer):
    def __init__(self, *args, num_groups, eta=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_groups = num_groups
        self.eta = eta
        self.group_weights = torch.ones(num_groups) / num_groups
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        group_ids = inputs.pop('group_id')
        sample_weights = inputs.pop('sample_weight', None)
        
        if self.group_weights.device != labels.device:
            self.group_weights = self.group_weights.to(labels.device)
        
        outputs = model(**inputs)
        logits = outputs.logits
        
        loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
        per_sample_loss = loss_fct(logits, labels)
        
        if sample_weights is not None:
            per_sample_loss = per_sample_loss * sample_weights.to(labels.device)
        
        group_losses = torch.zeros(self.num_groups, device=logits.device)
        group_counts = torch.zeros(self.num_groups, device=logits.device)
        
        for g in range(self.num_groups):
            mask = (group_ids == g)
            if mask.sum() > 0:
                group_losses[g] = per_sample_loss[mask].mean()
                group_counts[g] = mask.sum()
        
        with torch.no_grad():
            active_mask = (group_counts > 0)
            if active_mask.sum() > 0:
                self.group_weights[active_mask] *= torch.exp(self.eta * group_losses[active_mask])
                self.group_weights /= self.group_weights.sum()
        
        active_weights = self.group_weights[active_mask]
        active_losses = group_losses[active_mask]
        
        if active_weights.sum() > 0:
            loss = (active_weights * active_losses).sum() / active_weights.sum()
        else:
            loss = per_sample_loss.mean()
        
        return (loss, outputs) if return_outputs else loss

In [14]:
# === CHECK FOR CHECKPOINT ===
checkpoint_dir = None
checkpoints = list(OUTPUT_DIR.glob('checkpoint-*'))
if checkpoints:
    latest = max(checkpoints, key=lambda x: int(x.name.split('-')[1]))
    checkpoint_dir = str(latest)
    print(f'Resuming from: {checkpoint_dir}')
else:
    print('Starting fresh')

Starting fresh


In [17]:
# === TRAINING ===
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
model = model.to(device)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    load_best_model_at_end=True,
    logging_steps=50,
    seed=RANDOM_STATE,
    report_to='none',
    dataloader_num_workers=0,
    fp16=False,
    remove_unused_columns=False,
)

trainer = GroupDROTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=GroupDROCollator(),
    num_groups=num_groups,

    eta=GROUPDRO_ETA,
)

print(f'Training on {device}...')
trainer.train(resume_from_checkpoint=checkpoint_dir)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1297.78it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those p

Training on mps...


Epoch,Training Loss,Validation Loss
1,0.253725,1.301737
2,0.248925,1.225357
3,0.255173,1.233632
4,0.231515,1.303648
5,0.183753,1.401461
6,0.184755,1.478696
7,0.173128,1.579564
8,0.226555,1.488889
9,0.184915,1.636166
10,0.134860,1.678249


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.04it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

TrainOutput(global_step=20670, training_loss=0.21159454633143424, metrics={'train_runtime': 38321.8989, 'train_samples_per_second': 4.313, 'train_steps_per_second': 0.539, 'total_flos': 1.08737477367552e+16, 'train_loss': 0.21159454633143424, 'epoch': 10.0})

In [18]:
# === SAVE MODEL ===
final_dir = OUTPUT_DIR / 'final'
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
joblib.dump(le, final_dir / 'label_encoder.joblib')
print(f'Saved to: {final_dir}')

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.29it/s]

Saved to: ../output/combined_scrubbing_groupdro/final


In [19]:
# === EVALUATION ===
model.eval()
dataloader = DataLoader(test_dataset, batch_size=32, collate_fn=GroupDROCollator())
all_preds, all_labels = [], []

print('Evaluating...')
with torch.no_grad():
    for batch in tqdm(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        all_preds.extend(outputs.logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(batch['labels'].numpy())

y_true, y_pred = np.array(all_labels), np.array(all_preds)
print(f'Accuracy: {accuracy_score(y_true, y_pred):.4f}')
print(f'Macro F1: {f1_score(y_true, y_pred, average="macro"):.4f}')

Evaluating...


100%|██████████| 173/173 [02:55<00:00,  1.02s/it]

Accuracy: 0.5762
Macro F1: 0.5970


In [20]:
# === FAIRNESS METRICS ===
if group_col:
    df_test['y_true'] = y_true
    df_test['y_pred'] = y_pred
    
    def compute_tpr_gaps(df, group_col, num_classes, min_support=30):
        groups = sorted(df[group_col].dropna().unique())
        tpr_matrix = np.full((len(groups), num_classes), np.nan)
        
        for gi, g in enumerate(groups):
            df_g = df[df[group_col] == g]
            for c in range(num_classes):
                pos_mask = (df_g['y_true'] == c)
                support = pos_mask.sum()
                if support >= min_support:
                    tp = ((df_g['y_pred'] == c) & pos_mask).sum()
                    tpr_matrix[gi, c] = tp / support
        
        gap_per_class = []
        for c in range(num_classes):
            col = tpr_matrix[:, c]
            valid = col[~np.isnan(col)]
            gap_per_class.append(np.max(valid) - np.min(valid) if len(valid) >= 2 else np.nan)
        
        valid_gaps = [g for g in gap_per_class if not np.isnan(g)]
        return max(valid_gaps) if valid_gaps else np.nan, np.mean(valid_gaps) if valid_gaps else np.nan
    
    worst_gap, macro_gap = compute_tpr_gaps(df_test, group_col, num_labels)
    print(f'Worst TPR Gap: {worst_gap:.3f}')
    print(f'Macro TPR Gap: {macro_gap:.3f}')
else:
    worst_gap, macro_gap = np.nan, np.nan

Worst TPR Gap: 0.324
Macro TPR Gap: 0.103


In [21]:
# === FINAL COMPARISON ===
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='macro')

print('=' * 65)
print('COMPARISON: Combined Method vs Paper Results')
print('=' * 65)
print(f'{"Method":<35} {"Acc":>7} {"F1":>7} {"Worst":>7} {"Macro":>7}')
print('-' * 65)
print(f'{"Baseline (paper)":<35} {"0.609":>7} {"0.621":>7} {"0.329":>7} {"0.116":>7}')
print(f'{"GroupDRO (paper)":<35} {"0.548":>7} {"0.553":>7} {"0.265":>7} {"0.113":>7}')
print(f'{"Data Scrubbing (paper)":<35} {"0.594":>7} {"0.611":>7} {"0.300":>7} {"0.112":>7}')
print('-' * 65)
if not np.isnan(worst_gap):
    print(f'{"Combined (NEW)":<35} {acc:>7.3f} {f1:>7.3f} {worst_gap:>7.3f} {macro_gap:>7.3f}')
else:
    print(f'{"Combined (NEW)":<35} {acc:>7.3f} {f1:>7.3f} {"N/A":>7} {"N/A":>7}')
print('=' * 65)
print('Done!')

COMPARISON: Combined Method vs Paper Results
Method                                  Acc      F1   Worst   Macro
-----------------------------------------------------------------
Baseline (paper)                      0.609   0.621   0.329   0.116
GroupDRO (paper)                      0.548   0.553   0.265   0.113
Data Scrubbing (paper)                0.594   0.611   0.300   0.112
-----------------------------------------------------------------
Combined (NEW)                        0.576   0.597   0.324   0.103
Done!
